# 🔬 LSHBloom — Chạy trên data của bạn

Notebook này chạy **LSHBloom** trên dataset benchmark của bài báo với format:

| Cột | Ý nghĩa |
|---|---|
| `doc_id` | ID duy nhất của mỗi document |
| `source_doc_id` | ID document gốc — **2 docs cùng source_doc_id = duplicate** |
| `split` | `tuning` hoặc `test` |
| `variant_family` | Loại biến thể (`original`, ...) |
| `variant_type` | Kiểu duplicate (`original`, `parse_noise`, `truncation`, ...) |
| `text` | Nội dung văn bản |
| `n_words` | Số từ |

**Nhãn ground truth:** `doc_id != source_doc_id` → duplicate (label = 1)

## 📦 Bước 1 — Cài thư viện

In [1]:
!pip install datasketch pybloom-live mmh3 -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 12.9 MB/s eta 0:00:00


## 📚 Bước 2 — Import

In [2]:
import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm import tqdm

import mmh3
from datasketch import MinHash, MinHashLSH
from pybloom_live import BloomFilter
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print('✅ Thư viện sẵn sàng!')

✅ Thư viện sẵn sàng!


## 📥 Bước 3 — Đọc data

⚠️ **Thay đường dẫn file cho đúng với Kaggle của bạn!**

In [8]:
# ── Thay đường dẫn này cho đúng ───────────────────────────────────────────
TUNING_PATH = '/kaggle/input/datasets/namthanh189/dataset-lshbloom/data/tuning_results.csv'
TEST_PATH   = '/kaggle/input/datasets/namthanh189/dataset-lshbloom/data/test_results.csv'
# ──────────────────────────────────────────────────────────────────────────

df_tuning = pd.read_csv(TUNING_PATH)
df_test   = pd.read_csv(TEST_PATH)

print(f'✅ Tuning : {len(df_tuning):,} docs')
print(f'✅ Test   : {len(df_test):,} docs')
print(f'\nCác cột: {list(df_tuning.columns)}')
print(f'\nVí dụ 3 dòng đầu:')
df_tuning.head(3)

✅ Tuning : 6 docs
✅ Test   : 3 docs

Các cột: ['ngram_n', 'num_perm', 'threshold', 'mean_precision', 'mean_recall', 'mean_f1', 'min_f1', 'max_f1', 'mean_query_sec', 'mean_insert_sec']

Ví dụ 3 dòng đầu:


,ngram_n,num_perm,threshold,mean_precision,mean_recall,mean_f1,min_f1,max_f1,mean_query_sec,mean_insert_sec
0,3,128,0.4,1.0,0.999333,0.999666,0.999500,1.000000,0.101638,0.241397
1,3,64,0.4,1.0,0.994111,0.997046,0.996488,0.997912,0.054737,0.113165
2,3,128,0.6,1.0,0.953111,0.975969,0.970664,0.982620,0.063659,0.226735


In [12]:
# Chạy cell này TRƯỚC add_labels
df_tuning = pd.read_parquet('/kaggle/input/datasets/namthanh189/dataset-lshbloom/data/tuning_docs.parquet')
df_test   = pd.read_parquet('/kaggle/input/datasets/namthanh189/dataset-lshbloom/data/test_docs.parquet')

print("Tuning shape:", df_tuning.shape)
print("Tuning columns:", df_tuning.columns.tolist())

Tuning shape: (21000, 7)
Tuning columns: ['doc_id', 'source_doc_id', 'split', 'variant_family', 'variant_type', 'text', 'n_words']


## 🏷️ Bước 4 — Tạo nhãn ground truth

Quy tắc từ bài báo: nếu `doc_id != source_doc_id` → đây là bản sao (duplicate = 1)

In [13]:
def add_labels(df):
    """Tạo nhãn: 1 = duplicate, 0 = original"""
    df = df.copy()
    df['label'] = (df['doc_id'] != df['source_doc_id']).astype(int)
    return df

df_tuning = add_labels(df_tuning)
df_test   = add_labels(df_test)

for name, df in [('Tuning', df_tuning), ('Test', df_test)]:
    n_orig = (df['label'] == 0).sum()
    n_dup  = (df['label'] == 1).sum()
    print(f'{name}: {len(df):,} docs | '
          f'Original={n_orig:,} ({n_orig/len(df)*100:.1f}%) | '
          f'Duplicate={n_dup:,} ({n_dup/len(df)*100:.1f}%)')

print(f'\nPhân phối variant_type trong Test:')
print(df_test['variant_type'].value_counts().to_string())

Tuning: 21,000 docs | Original=3,000 (14.3%) | Duplicate=18,000 (85.7%)
Test: 70,000 docs | Original=10,000 (14.3%) | Duplicate=60,000 (85.7%)

Phân phối variant_type trong Test:
variant_type
original                  10000
parser_noise_v0           10000
parser_noise_v1           10000
parser_noise_v2           10000
truncate_head_v2           2032
truncate_two_gaps_v1       2019
truncate_three_gaps_v0     2017
truncate_middle_v1         2007
truncate_tail_v0           2003
truncate_tail_v2           2003
truncate_head_v1           1992
truncate_two_gaps_v0       1982
truncate_middle_v2         1981
truncate_middle_v0         1979
truncate_two_gaps_v2       1978
truncate_tail_v1           1953
truncate_three_gaps_v1     1946
truncate_head_v0           1936
truncate_three_gaps_v2     1923
truncate_none_v0             83
truncate_none_v1             83
truncate_none_v2             83


## ⚙️ Bước 5 — MinHash helper

In [14]:
def get_shingles(text, n=3):
    """Word n-gram shingles"""
    if not isinstance(text, str):
        return set()
    tokens = re.sub(r'[^a-z0-9\s]', '', text.lower()).split()
    if len(tokens) < n:
        return set(tokens)
    return {' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)}

def make_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in get_shingles(text):
        m.update(s.encode('utf-8'))
    return m

print('✅ MinHash helper sẵn sàng!')

✅ MinHash helper sẵn sàng!


## 🔵 Bước 6 — LSHBloom

In [15]:
class LSHBloom:
    """
    LSHBloom: datasketch bands + pybloom_live index + mmh3 hashing.
    """
    def __init__(self, num_perm=128, threshold=0.5, n_expected=10000, fp_rate=1e-5):
        self.num_perm  = num_perm
        self.threshold = threshold

        _lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
        self.hashranges = _lsh.hashranges
        self.b = len(self.hashranges)
        del _lsh

        self.blooms = [
            BloomFilter(capacity=n_expected, error_rate=fp_rate)
            for _ in range(self.b)
        ]
        print(f'  LSHBloom: {self.b} bands | threshold={threshold} | fp_rate={fp_rate}')

    def _band_key(self, minhash, i):
        start, end    = self.hashranges[i]
        band_bytes    = minhash.hashvalues[start:end].astype(np.uint64).tobytes()
        return str(mmh3.hash128(band_bytes, signed=False))

    def query(self, minhash):
        return any(self._band_key(minhash, i) in self.blooms[i]
                   for i in range(self.b))

    def insert(self, minhash):
        for i in range(self.b):
            self.blooms[i].add(self._band_key(minhash, i))

    def query_and_insert(self, minhash):
        is_dup = self.query(minhash)
        self.insert(minhash)
        return is_dup

    def index_size_mb(self):
        return sum(b.num_bits for b in self.blooms) / 8 / 1024 / 1024


print('✅ LSHBloom sẵn sàng!')

✅ LSHBloom sẵn sàng!


## 🎯 Bước 7 — Tuning: tìm threshold tốt nhất

Dùng **tuning set** để chọn threshold cho F1 cao nhất — đúng theo quy trình bài báo.

In [16]:
def run_lshbloom(df, num_perm=128, threshold=0.5):
    """Chạy LSHBloom trên một DataFrame, trả về predictions và metrics."""
    model  = LSHBloom(num_perm=num_perm, threshold=threshold,
                      n_expected=len(df), fp_rate=1e-5)
    preds  = []
    pairs  = []   # [(orig_doc_id, dup_doc_id, orig_text, dup_text), ...]
    seen   = {}   # {band_key_tuple: doc_id} — để tra cặp

    # Index phụ để tìm cặp duplicate cụ thể
    pair_lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    id_to_row = {}

    t0 = time.time()
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
        text   = str(row['text']) if pd.notna(row['text']) else ''
        doc_id = str(row['doc_id'])
        mh     = make_minhash(text, num_perm)

        is_dup = model.query_and_insert(mh)
        preds.append(int(is_dup))

        if is_dup:
            matches = pair_lsh.query(mh)
            if matches:
                orig_id  = matches[0]
                orig_row = id_to_row.get(orig_id, {})
                pairs.append({
                    'orig_doc_id'    : orig_id,
                    'orig_source_id' : orig_row.get('source_doc_id', ''),
                    'orig_type'      : orig_row.get('variant_type', ''),
                    'orig_text'      : str(orig_row.get('text', ''))[:300],
                    'dup_doc_id'     : doc_id,
                    'dup_source_id'  : row.get('source_doc_id', ''),
                    'dup_type'       : row.get('variant_type', ''),
                    'dup_text'       : text[:300],
                    'same_source'    : orig_row.get('source_doc_id', '') == row.get('source_doc_id', ''),
                })

        try:
            pair_lsh.insert(doc_id, mh)
            id_to_row[doc_id] = row.to_dict()
        except ValueError:
            pass

    elapsed = time.time() - t0
    labels  = df['label'].tolist()

    p  = precision_score(labels, preds, zero_division=0)
    r  = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    return {
        'threshold' : threshold,
        'num_perm'  : num_perm,
        'precision' : p,
        'recall'    : r,
        'f1'        : f1,
        'time_sec'  : elapsed,
        'index_mb'  : model.index_size_mb(),
        'preds'     : preds,
        'labels'    : labels,
        'pairs'     : pairs,
    }


# ── Chạy tuning: thử nhiều threshold ──
THRESHOLDS = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]
NUM_PERM   = 128

print('🎯 Tuning LSHBloom trên tuning set...')
tuning_results = []
for t in THRESHOLDS:
    print(f'\n  Threshold = {t}')
    r = run_lshbloom(df_tuning, num_perm=NUM_PERM, threshold=t)
    tuning_results.append(r)
    print(f'  P={r["precision"]:.4f}  R={r["recall"]:.4f}  F1={r["f1"]:.4f}  '
          f'({r["time_sec"]:.1f}s)')

df_tuning_res = pd.DataFrame([{
    'threshold': r['threshold'], 'precision': r['precision'],
    'recall': r['recall'], 'f1': r['f1'], 'time_sec': r['time_sec']
} for r in tuning_results])

best = df_tuning_res.loc[df_tuning_res['f1'].idxmax()]
BEST_THRESHOLD = best['threshold']

print(f'\n✅ Threshold tốt nhất: {BEST_THRESHOLD} (F1 = {best["f1"]:.4f})')
print(df_tuning_res.to_string(index=False))

🎯 Tuning LSHBloom trên tuning set...

  Threshold = 0.3
  LSHBloom: 37 bands | threshold=0.3 | fp_rate=1e-05


  P=0.9996  R=1.0000  F1=0.9998  (416.0s)

  Threshold = 0.5
  LSHBloom: 25 bands | threshold=0.5 | fp_rate=1e-05


  P=1.0000  R=0.9951  F1=0.9975  (397.1s)

  Threshold = 0.6
  LSHBloom: 18 bands | threshold=0.6 | fp_rate=1e-05


  P=1.0000  R=0.9638  F1=0.9816  (397.7s)

  Threshold = 0.7
  LSHBloom: 14 bands | threshold=0.7 | fp_rate=1e-05


  P=1.0000  R=0.8967  F1=0.9455  (380.7s)

  Threshold = 0.8
  LSHBloom: 9 bands | threshold=0.8 | fp_rate=1e-05


  P=1.0000  R=0.7349  F1=0.8472  (398.3s)

  Threshold = 0.9
  LSHBloom: 5 bands | threshold=0.9 | fp_rate=1e-05


  P=1.0000  R=0.5231  F1=0.6869  (399.0s)

✅ Threshold tốt nhất: 0.3 (F1 = 0.9998)
 threshold  precision   recall       f1   time_sec
       0.3   0.999556 1.000000 0.999778 415.989314
       0.5   1.000000 0.995056 0.997522 397.125589
       0.6   1.000000 0.963833 0.981584 397.661141
       0.7   1.000000 0.896722 0.945549 380.673725
       0.8   1.000000 0.734944 0.847225 398.267154
       0.9   1.000000 0.523111 0.686898 399.039386


## ▶️ Bước 8 — Test: chạy với threshold tốt nhất

In [17]:
print(f'🔵 Chạy LSHBloom trên test set (threshold={BEST_THRESHOLD})...')
test_result = run_lshbloom(df_test, num_perm=NUM_PERM, threshold=BEST_THRESHOLD)

print(f'\n✅ Kết quả trên test set:')
print(f'   Precision : {test_result["precision"]:.4f}')
print(f'   Recall    : {test_result["recall"]:.4f}')
print(f'   F1 Score  : {test_result["f1"]:.4f}')
print(f'   Thời gian : {test_result["time_sec"]:.1f}s')
print(f'   Index size: {test_result["index_mb"]:.3f} MB')
print(f'   Số cặp duplicate tìm thấy: {len(test_result["pairs"]):,}')

🔵 Chạy LSHBloom trên test set (threshold=0.3)...
  LSHBloom: 37 bands | threshold=0.3 | fp_rate=1e-05



✅ Kết quả trên test set:
   Precision : 0.9989
   Recall    : 0.9999
   F1 Score  : 0.9994
   Thời gian : 1263.8s
   Index size: 7.399 MB
   Số cặp duplicate tìm thấy: 60,065


## 🔍 Bước 9 — Xem các cặp Duplicate cụ thể

In [18]:
def show_pairs(pairs, n_show=5, only_correct=False):
    """
    Hiển thị các cặp duplicate.
    only_correct=True → chỉ hiện cặp đúng (same_source=True)
    """
    if only_correct:
        pairs = [p for p in pairs if p['same_source']]
        label = '✅ ĐÚNG (cùng source_doc_id)'
    else:
        label = 'TẤT CẢ'

    print(f'\n{"-"*70}')
    print(f'  {len(pairs):,} cặp duplicate — hiện {min(n_show, len(pairs))} cặp [{label}]')
    print(f'{"-"*70}')

    for k, p in enumerate(pairs[:n_show], 1):
        correct = '✅' if p['same_source'] else '❌ FALSE POSITIVE'
        print(f'\n┌─ Cặp #{k} {correct}')
        print(f'│  📄 ORIGINAL  doc_id={p["orig_doc_id"]} | type={p["orig_type"]}')
        print(f'│     {p["orig_text"][:200].replace(chr(10), " ")}...')
        print(f'│')
        print(f'│  🔁 DUPLICATE doc_id={p["dup_doc_id"]} | type={p["dup_type"]}')
        print(f'│     {p["dup_text"][:200].replace(chr(10), " ")}...')
        print(f'│')
        print(f'│  source_doc_id: orig={p["orig_source_id"]} | dup={p["dup_source_id"]}')
        print(f'└{"─"*68}')


# Xem 5 cặp đúng (True Positive)
show_pairs(test_result['pairs'], n_show=5, only_correct=True)


----------------------------------------------------------------------
  59,761 cặp duplicate — hiện 5 cặp [✅ ĐÚNG (cùng source_doc_id)]
----------------------------------------------------------------------

┌─ Cặp #1 ✅
│  📄 ORIGINAL  doc_id=259317121 | type=original
│     Variational Bayesian experimental design for geophysical applications: seismic source location, amplitude versus offset inversion, and estimating CO2 saturations in a subsurface reservoir This paper i...
│
│  🔁 DUPLICATE doc_id=259317121_p0 | type=parser_noise_v0
│     Variational Bayesian experimental design for geophysical applications: seismic source location, amplitude versus offset inversion, and estimating CO2 saturations in a subsurface reservoir This paper i...
│
│  source_doc_id: orig=259317121 | dup=259317121
└────────────────────────────────────────────────────────────────────

┌─ Cặp #2 ✅
│  📄 ORIGINAL  doc_id=259317121 | type=original
│     Variational Bayesian experimental design for geophysical appli

In [19]:
# Xem 5 cặp False Positive (phát hiện nhầm)
show_pairs(test_result['pairs'], n_show=5, only_correct=False)


----------------------------------------------------------------------
  60,065 cặp duplicate — hiện 5 cặp [TẤT CẢ]
----------------------------------------------------------------------

┌─ Cặp #1 ✅
│  📄 ORIGINAL  doc_id=259317121 | type=original
│     Variational Bayesian experimental design for geophysical applications: seismic source location, amplitude versus offset inversion, and estimating CO2 saturations in a subsurface reservoir This paper i...
│
│  🔁 DUPLICATE doc_id=259317121_p0 | type=parser_noise_v0
│     Variational Bayesian experimental design for geophysical applications: seismic source location, amplitude versus offset inversion, and estimating CO2 saturations in a subsurface reservoir This paper i...
│
│  source_doc_id: orig=259317121 | dup=259317121
└────────────────────────────────────────────────────────────────────

┌─ Cặp #2 ✅
│  📄 ORIGINAL  doc_id=259317121 | type=original
│     Variational Bayesian experimental design for geophysical applications: seismic sour

## 📊 Bước 10 — Phân tích theo variant_type

In [ ]:
# Gắn prediction vào df_test
df_test = df_test.copy()
df_test['pred'] = test_result['preds']

# F1 theo từng variant_type
print('📊 Kết quả theo variant_type:')
rows = []
for vtype, grp in df_test.groupby('variant_type'):
    if grp['label'].nunique() < 2:
        continue
    p  = precision_score(grp['label'], grp['pred'], zero_division=0)
    r  = recall_score(grp['label'], grp['pred'], zero_division=0)
    f1 = f1_score(grp['label'], grp['pred'], zero_division=0)
    rows.append({'variant_type': vtype, 'n': len(grp),
                 'precision': p, 'recall': r, 'f1': f1})

df_vtype = pd.DataFrame(rows).sort_values('f1', ascending=False)
print(df_vtype.to_string(index=False))

## 🎨 Bước 11 — Biểu đồ kết quả

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('LSHBloom — Kết quả trên data của bạn', fontsize=16, fontweight='bold')

BLUE = '#1976D2'

# 1. F1 / Precision / Recall theo Threshold (tuning)
ax = axes[0, 0]
ax.plot(df_tuning_res['threshold'], df_tuning_res['f1'],
        marker='o', color=BLUE, lw=2.5, ms=8, label='F1')
ax.plot(df_tuning_res['threshold'], df_tuning_res['precision'],
        marker='s', color='#43A047', lw=2, ms=7, linestyle='--', label='Precision')
ax.plot(df_tuning_res['threshold'], df_tuning_res['recall'],
        marker='^', color='#E64A19', lw=2, ms=7, linestyle=':', label='Recall')
ax.axvline(BEST_THRESHOLD, color='gray', linestyle='--', alpha=0.6,
           label=f'Best T={BEST_THRESHOLD}')
ax.set(title='Metrics theo Threshold (Tuning set)',
       xlabel='Threshold', ylabel='Score')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=0.3)

# 2. Confusion Matrix trên Test set
ax = axes[0, 1]
cm = confusion_matrix(test_result['labels'], test_result['preds'])
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred: Original', 'Pred: Duplicate'])
ax.set_yticklabels(['True: Original', 'True: Duplicate'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title('Confusion Matrix (Test set)', fontweight='bold')
plt.colorbar(im, ax=ax)

# 3. F1 theo variant_type
ax = axes[0, 2]
if len(df_vtype) > 0:
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(df_vtype)))
    bars = ax.barh(df_vtype['variant_type'], df_vtype['f1'],
                   color=colors, edgecolor='white')
    for bar, val in zip(bars, df_vtype['f1']):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)
ax.set(title='F1 theo Variant Type (Test set)',
       xlabel='F1 Score', xlim=(0, 1.15))
ax.grid(alpha=0.3, axis='x')

# 4. Precision / Recall / F1 — bar chart tổng kết
ax = axes[1, 0]
metrics = ['Precision', 'Recall', 'F1']
values  = [test_result['precision'], test_result['recall'], test_result['f1']]
bars = ax.bar(metrics, values, color=[BLUE, '#43A047', '#E64A19'],
              width=0.5, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
ax.set(title=f'Kết quả Test Set (T={BEST_THRESHOLD})', ylim=(0, 1.15))
ax.grid(alpha=0.3, axis='y')

# 5. Phân phối n_words: original vs duplicate
ax = axes[1, 1]
df_orig = df_test[df_test['label'] == 0]['n_words'].clip(upper=2000)
df_dup  = df_test[df_test['label'] == 1]['n_words'].clip(upper=2000)
ax.hist(df_orig, bins=40, alpha=0.6, color='#43A047', label='Original')
ax.hist(df_dup,  bins=40, alpha=0.6, color=BLUE,     label='Duplicate')
ax.set(title='Phân phối độ dài (n_words)',
       xlabel='Số từ (clip tại 2000)', ylabel='Số tài liệu')
ax.legend(); ax.grid(alpha=0.3)

# 6. True/False Positive/Negative breakdown
ax = axes[1, 2]
tn, fp, fn, tp = cm.ravel()
labels_pie = ['True Neg\n(correct orig)', 'False Pos\n(wrong dup)',
               'False Neg\n(missed dup)', 'True Pos\n(correct dup)']
values_pie = [tn, fp, fn, tp]
colors_pie = ['#A5D6A7', '#EF9A9A', '#FFCC80', '#90CAF9']
wedges, texts, autotexts = ax.pie(
    values_pie, labels=labels_pie, colors=colors_pie,
    autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(9)
ax.set_title('Phân tích TP/TN/FP/FN', fontweight='bold')

plt.tight_layout()
plt.savefig('lshbloom_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Biểu đồ đã lưu!')

## 💾 Bước 12 — Export kết quả

In [ ]:
# Export predictions
df_out = df_test[['doc_id', 'source_doc_id', 'variant_type', 'label']].copy()
df_out['lshbloom_pred'] = test_result['preds']
df_out['correct']       = (df_out['label'] == df_out['lshbloom_pred'])
df_out.to_csv('lshbloom_predictions.csv', index=False)
print(f'✅ Predictions → lshbloom_predictions.csv ({len(df_out):,} rows)')

# Export duplicate pairs
df_pairs = pd.DataFrame(test_result['pairs'])[
    ['orig_doc_id', 'orig_source_id', 'orig_type', 'orig_text',
     'dup_doc_id',  'dup_source_id',  'dup_type',  'dup_text', 'same_source']
]
df_pairs.to_csv('lshbloom_duplicate_pairs.csv', index=False)
print(f'✅ Duplicate pairs → lshbloom_duplicate_pairs.csv ({len(df_pairs):,} cặp)')

# Tổng kết
print(f'\n{"="*60}')
print(f'  TỔNG KẾT LSHBLOOM TRÊN DATA CỦA BẠN')
print(f'{"="*60}')
print(f'  Best threshold : {BEST_THRESHOLD}')
print(f'  Precision      : {test_result["precision"]:.4f}')
print(f'  Recall         : {test_result["recall"]:.4f}')
print(f'  F1 Score       : {test_result["f1"]:.4f}')
print(f'  Thời gian      : {test_result["time_sec"]:.1f}s')
print(f'  Index size     : {test_result["index_mb"]:.3f} MB')
tp_pairs = sum(1 for p in test_result['pairs'] if p['same_source'])
fp_pairs = sum(1 for p in test_result['pairs'] if not p['same_source'])
print(f'  Cặp đúng (TP)  : {tp_pairs:,}')
print(f'  Cặp sai (FP)   : {fp_pairs:,}')
print(f'{"="*60}')